# Text-Grounded Import Contract

This notebook proves the Phase 4 contract slice before any `llm_client`
extractor is wired in. It shows that `onto-canon6` can persist source
text, claim text, and exact evidence spans on a candidate assertion, and
fail loudly when a proposed span does not match the source.


In [1]:
from pathlib import Path
import shutil

from onto_canon6.pipeline import (
    CandidateAssertionImport,
    EvidenceSpan,
    ProfileRef,
    ReviewService,
    SourceArtifactRef,
)

runtime_dir = Path('../var/notebook_phase4_import').resolve()
if runtime_dir.exists():
    shutil.rmtree(runtime_dir)
runtime_dir.mkdir(parents=True)

service = ReviewService(
    db_path=runtime_dir / 'review.sqlite3',
    overlay_root=runtime_dir / 'ontology_overlays',
)


In [2]:
source_text = 'Mission planning uses the radar system during the exercise.'

candidate_import = CandidateAssertionImport(
    profile=ProfileRef(profile_id='default', profile_version='1.0.0'),
    payload={
        'predicate': 'oc:uses_system_demo',
        'roles': {
            'subject': [{'entity_id': 'ent:activity:mission_planning'}],
            'object': [{'entity_id': 'ent:system:radar_system'}],
        },
    },
    submitted_by='notebook:phase4',
    source_artifact=SourceArtifactRef(
        source_kind='raw_text',
        source_ref='text://phase4/mission-planning',
        source_label='phase4 notebook raw text',
        source_metadata={'notebook': '07_text_grounded_import_contract'},
        content_text=source_text,
    ),
    evidence_spans=(
        EvidenceSpan(start_char=0, end_char=16, text='Mission planning'),
        EvidenceSpan(start_char=26, end_char=38, text='radar system'),
    ),
    claim_text='Mission planning uses the radar system.',
)

result = service.submit_candidate_import(candidate_import=candidate_import)
result.candidate.model_dump()


{'candidate_id': 'cand_5ca69eb2ed9d490fa58d0808',
 'profile': {'profile_id': 'default', 'profile_version': '1.0.0'},
 'validation_status': 'valid',
 'review_status': 'pending_review',
 'payload_hash': 'sha256:c89ee57bf439466fdd08ce7eb685f4470776ec3f28aa0fbdeb86d3a5ee239a63',
 'payload': {'predicate': 'oc:uses_system_demo',
  'roles': {'object': [{'entity_id': 'ent:system:radar_system',
     'kind': 'entity'}],
   'subject': [{'entity_id': 'ent:activity:mission_planning',
     'kind': 'entity'}]}},
 'normalized_payload': {'predicate': 'oc:uses_system_demo',
  'roles': {'object': [{'entity_id': 'ent:system:radar_system',
     'kind': 'entity'}],
   'subject': [{'entity_id': 'ent:activity:mission_planning',
     'kind': 'entity'}]}},
 'validation': {'hard_errors': (),
  'soft_violations': (),
  'type_checks_total': 0,
  'type_checks_unknown': 0},
 'proposal_ids': (),
 'provenance': {'submitted_by': 'notebook:phase4',
  'source_artifact': {'source_kind': 'raw_text',
   'source_ref': 'text:

In [3]:
persisted = service.list_candidate_assertions()
persisted[0].claim_text, persisted[0].evidence_spans, persisted[0].provenance.content_text


('Mission planning uses the radar system.',
 (EvidenceSpan(start_char=0, end_char=16, text='Mission planning'),
  EvidenceSpan(start_char=26, end_char=38, text='radar system')),
 'Mission planning uses the radar system during the exercise.')

In [4]:
bad_import = CandidateAssertionImport(
    profile=ProfileRef(profile_id='default', profile_version='1.0.0'),
    payload={
        'predicate': 'oc:uses_system_demo',
        'roles': {'subject': [{'entity_id': 'ent:activity:mission_planning'}]},
    },
    submitted_by='notebook:phase4',
    source_artifact=SourceArtifactRef(
        source_kind='raw_text',
        source_ref='text://phase4/bad-span',
        content_text='Alpha Beta',
    ),
    evidence_spans=(EvidenceSpan(start_char=0, end_char=5, text='Wrong'),),
)

try:
    service.submit_candidate_import(candidate_import=bad_import)
except ValueError as exc:
    str(exc)
